# ⬢ Lab 05 — AI Agents, LLMOps & MLflow Tracing
**Build a ReAct agent, trace every step with MLflow, measure cost and quality**

---
**Real-world scenario:** Adyen (Amsterdam) processes 1B+ payment transactions per year.
When a transaction is flagged as suspicious, an analyst must check merchant history, device fingerprints,
IP geolocation, and customer patterns — manually taking 15-20 minutes per case.
You will build the AI agent that does this in seconds, with full MLflow tracing.

**What you will build:**
1. ReAct from scratch (understand the loop before using a framework)
2. A LangGraph multi-step agent with real tools
3. MLflow tracing — trace every LLM call, tool invocation, cost, and latency
4. Evaluate agent quality with a LLM-as-judge
5. Cost tracking and optimisation analysis

**Estimated time:** 65 min | **Level:** Advanced | **MLflow: built-in on Databricks**

In [ ]:
%pip install -q openai langgraph langchain langchain-openai mlflow pandas

In [ ]:
import os, json, time, re
import pandas as pd
import mlflow
from openai import OpenAI

client = OpenAI()
mlflow.set_experiment('Adyen-Fraud-Agent')
mlflow.openai.autolog()   # auto-traces every OpenAI call — zero extra code

print('MLflow OpenAI autolog enabled. Every LLM call will be traced automatically.')

## Part 1 — ReAct from Scratch

ReAct = **Re**ason + **Act**. The loop: think about what to do → call a tool → observe the result → think again → repeat until done. Understanding this loop is essential before using LangGraph.

In [ ]:
# Simulated tools (in production these call real databases/APIs)
def get_merchant_fraud_rate(merchant_id: str) -> str:
    data = {
        'MERCH_001': {'name': 'ElectroMax NL', 'fraud_rate': 0.003, 'monthly_volume': 85000, 'category': 'electronics'},
        'MERCH_002': {'name': 'QuickCash Exchange', 'fraud_rate': 0.089, 'monthly_volume': 12000, 'category': 'money_transfer'},
        'MERCH_003': {'name': 'Albert Heijn Online', 'fraud_rate': 0.0008, 'monthly_volume': 2400000, 'category': 'grocery'},
    }
    m = data.get(merchant_id)
    if not m:
        return json.dumps({'error': 'Merchant not found'})
    return json.dumps(m)

def get_customer_recent_transactions(customer_id: str) -> str:
    data = {
        'CUST_4521': [
            {'amount': 45.20, 'merchant': 'Albert Heijn', 'country': 'NL', 'days_ago': 1},
            {'amount': 120.00, 'merchant': 'NS Reizen', 'country': 'NL', 'days_ago': 2},
            {'amount': 3200.00, 'merchant': 'ElectroMax NL', 'country': 'NL', 'days_ago': 0},
            {'amount': 18.50, 'merchant': 'Starbucks', 'country': 'NL', 'days_ago': 3},
        ]
    }
    txns = data.get(customer_id, [])
    return json.dumps({'transactions': txns, 'count': len(txns), 'max_single_txn': max((t['amount'] for t in txns), default=0)})

def check_ip_geolocation(ip_address: str) -> str:
    data = {
        '185.220.101.42': {'country': 'NL', 'city': 'Amsterdam', 'is_vpn': False, 'is_tor': False, 'risk_score': 0.12},
        '103.21.244.0':   {'country': 'CN', 'city': 'Shanghai',  'is_vpn': True,  'is_tor': False, 'risk_score': 0.87},
        '10.0.0.1':       {'country': 'NL', 'city': 'Rotterdam', 'is_vpn': False, 'is_tor': False, 'risk_score': 0.05},
    }
    return json.dumps(data.get(ip_address, {'error': 'IP not found', 'risk_score': 0.5}))

TOOLS = {
    'get_merchant_fraud_rate': get_merchant_fraud_rate,
    'get_customer_recent_transactions': get_customer_recent_transactions,
    'check_ip_geolocation': check_ip_geolocation,
}

TOOL_DESCRIPTIONS = '''Available tools (call as JSON: {"tool": "name", "args": {"param": "value"}}):
- get_merchant_fraud_rate(merchant_id): Returns merchant fraud rate, volume, category
- get_customer_recent_transactions(customer_id): Returns last 4 transactions for customer
- check_ip_geolocation(ip_address): Returns country, VPN/Tor detection, risk score
- FINAL_ANSWER: Call with {"tool": "FINAL_ANSWER", "risk_score": 0.0-1.0, "explanation": "..."}'''

REACT_SYSTEM = f'''You are a fraud analysis agent at Adyen. Investigate flagged transactions.

{TOOL_DESCRIPTIONS}

Format each response as either a tool call or a final answer. Think step by step.
Always check: merchant risk, customer history, and IP location before concluding.'''

def run_react_agent(transaction: dict, max_steps: int = 8) -> dict:
    messages = [
        {'role': 'system', 'content': REACT_SYSTEM},
        {'role': 'user', 'content': f'Investigate this transaction: {json.dumps(transaction)}'}
    ]
    steps, total_tokens = [], 0

    for step in range(max_steps):
        response = client.chat.completions.create(
            model='gpt-4o-mini',
            messages=messages,
            temperature=0,
        )
        content = response.choices[0].message.content
        total_tokens += response.usage.total_tokens
        messages.append({'role': 'assistant', 'content': content})

        # Parse JSON tool call
        try:
            call = json.loads(re.search(r'\{.*\}', content, re.DOTALL).group())
        except Exception:
            break

        if call.get('tool') == 'FINAL_ANSWER':
            return {'steps': steps, 'risk_score': call.get('risk_score', 0.5),
                    'explanation': call.get('explanation', ''), 'total_tokens': total_tokens}

        tool_fn = TOOLS.get(call.get('tool'))
        if tool_fn:
            args = call.get('args', {})
            result = tool_fn(**args)
            steps.append({'tool': call['tool'], 'args': args, 'result': result})
            messages.append({'role': 'user', 'content': f'Tool result: {result}'})
            print(f'  Step {step+1}: {call["tool"]}({args}) → {result[:80]}...')

    return {'steps': steps, 'risk_score': 0.5, 'explanation': 'Max steps reached', 'total_tokens': total_tokens}

# Test transaction
transaction = {
    'id': 'TXN-20260516-0042',
    'amount': 3200.00,
    'currency': 'EUR',
    'merchant_id': 'MERCH_001',
    'customer_id': 'CUST_4521',
    'ip_address': '185.220.101.42',
    'timestamp': '2026-05-16T14:32:00Z'
}

print(f'Investigating transaction: {transaction["id"]}\n')
with mlflow.start_run(run_name=f'fraud-analysis-{transaction["id"]}'):
    result = run_react_agent(transaction)
    mlflow.log_param('transaction_id', transaction['id'])
    mlflow.log_param('amount', transaction['amount'])
    mlflow.log_metric('risk_score', result['risk_score'])
    mlflow.log_metric('total_tokens', result['total_tokens'])
    mlflow.log_metric('agent_steps', len(result['steps']))

print(f'\nFinal Risk Score: {result["risk_score"]:.2f}')
print(f'Explanation: {result["explanation"]}')
print(f'Total tokens used: {result["total_tokens"]} (cost: ${result["total_tokens"]/1e6 * 0.60:.5f})')

## Part 2 — MLflow Traces: What Every Step Looked Like

Since we enabled `mlflow.openai.autolog()` at the start, every LLM call above was automatically traced.
Open the MLflow UI to see: full prompts, completions, token counts, latency per step.

On Databricks: Experiments tab → ING-RAG-Pipeline → click on the run → Traces

In [ ]:
# Custom spans: wrap your own business logic in MLflow traces
from mlflow.entities import SpanType

@mlflow.trace(name='fetch_risk_signals', span_type=SpanType.RETRIEVAL)
def fetch_all_risk_signals(transaction: dict) -> dict:
    merchant_data = json.loads(get_merchant_fraud_rate(transaction['merchant_id']))
    customer_data = json.loads(get_customer_recent_transactions(transaction['customer_id']))
    ip_data       = json.loads(check_ip_geolocation(transaction['ip_address']))
    return {'merchant': merchant_data, 'customer': customer_data, 'ip': ip_data}

@mlflow.trace(name='compute_risk_score', span_type=SpanType.CHAIN)
def compute_risk_score(signals: dict, transaction: dict) -> float:
    # Rule-based score as a baseline (deterministic, free, instant)
    score = 0.0
    if signals['merchant'].get('fraud_rate', 0) > 0.05:  score += 0.35
    if signals['ip'].get('is_vpn', False):               score += 0.30
    if signals['ip'].get('risk_score', 0) > 0.7:         score += 0.20
    if transaction['amount'] > signals['customer'].get('max_single_txn', 0) * 2:
        score += 0.15
    return min(score, 1.0)

# Run with custom spans
with mlflow.start_run(run_name='instrumented-analysis'):
    signals = fetch_all_risk_signals(transaction)    # traced as RETRIEVAL span
    score   = compute_risk_score(signals, transaction)  # traced as CHAIN span
    mlflow.log_metric('rule_based_risk_score', score)
    print(f'Rule-based risk score: {score:.2f}')
    print('Check MLflow UI to see the nested span tree.')

## Part 3 — LLM-as-Judge: Evaluate Agent Quality at Scale

In [ ]:
JUDGE_PROMPT = '''You are evaluating a fraud analysis agent's output.

Transaction: {transaction}
Agent explanation: {explanation}
Risk score: {risk_score}

Evaluate on three dimensions. Return valid JSON only:
{{
  "reasoning_quality": 1-5,
  "evidence_usage": 1-5,
  "decision_appropriateness": 1-5,
  "critique": "one sentence"
}}

Rubric:
- reasoning_quality: Does the explanation follow logically from evidence?
- evidence_usage: Were all available data points (merchant, customer, IP) considered?
- decision_appropriateness: Is the risk score consistent with the evidence?
5 = excellent, 1 = poor'''

def judge_agent_output(transaction: dict, agent_result: dict) -> dict:
    response = client.chat.completions.create(
        model='gpt-4o',   # use a more capable model as judge
        messages=[
            {'role': 'system', 'content': 'You are a precise evaluator. Return only valid JSON.'},
            {'role': 'user', 'content': JUDGE_PROMPT.format(
                transaction=json.dumps(transaction),
                explanation=agent_result['explanation'],
                risk_score=agent_result['risk_score']
            )}
        ],
        temperature=0,
        response_format={'type': 'json_object'}
    )
    return json.loads(response.choices[0].message.content)

with mlflow.start_run(run_name='agent-quality-eval'):
    judgment = judge_agent_output(transaction, result)
    mlflow.log_metrics({
        'judge_reasoning_quality':       judgment['reasoning_quality'],
        'judge_evidence_usage':          judgment['evidence_usage'],
        'judge_decision_appropriateness': judgment['decision_appropriateness'],
    })

    print('LLM Judge Evaluation:')
    print(f'  Reasoning quality:       {judgment["reasoning_quality"]}/5')
    print(f'  Evidence usage:          {judgment["evidence_usage"]}/5')
    print(f'  Decision appropriateness: {judgment["decision_appropriateness"]}/5')
    print(f'  Critique: {judgment["critique"]}')
    print('\nJudge scores logged to MLflow.')

## Part 4 — Cost Tracking and Optimisation

In [ ]:
# Cost per model (approximate May 2026)
MODEL_COSTS = {
    'gpt-4o':       {'input': 2.50, 'output': 10.00},   # $ per 1M tokens
    'gpt-4o-mini':  {'input': 0.15, 'output': 0.60},
    'claude-3-haiku-20240307': {'input': 0.25, 'output': 1.25},
}

def estimate_cost(input_tokens: int, output_tokens: int, model: str) -> float:
    costs = MODEL_COSTS.get(model, MODEL_COSTS['gpt-4o-mini'])
    return (input_tokens * costs['input'] + output_tokens * costs['output']) / 1_000_000

# Simulate cost for 1000 daily fraud investigations
avg_input_tokens  = 800   # system prompt + transaction + tool results
avg_output_tokens = 150   # agent reasoning + final answer
daily_volume      = 1000

print('Daily Cost Analysis (1,000 fraud investigations):')
print('-' * 55)
for model in MODEL_COSTS:
    cost_per_case = estimate_cost(avg_input_tokens, avg_output_tokens, model)
    daily_cost    = cost_per_case * daily_volume
    annual_cost   = daily_cost * 365
    print(f'{model:<35} ${cost_per_case:.5f}/case  ${annual_cost:,.0f}/yr')

print('\nOptimisation strategies:')
print('  1. Route simple cases (rule score < 0.1 or > 0.9) to rule-based only — no LLM needed')
print('  2. Cache tool results for same merchant/customer within 5 minutes')
print('  3. Use gpt-4o-mini for initial analysis, escalate to gpt-4o only for score 0.4-0.7')

# Estimate savings with smart routing
pct_simple = 0.65   # 65% of cases have obvious risk score (< 0.1 or > 0.9)
pct_complex = 1 - pct_simple
optimised_cost = daily_volume * (
    pct_simple * 0 +   # rule-based: free
    pct_complex * 0.6 * estimate_cost(avg_input_tokens, avg_output_tokens, 'gpt-4o-mini') +
    pct_complex * 0.4 * estimate_cost(avg_input_tokens, avg_output_tokens, 'gpt-4o')
) * 365

baseline_cost = estimate_cost(avg_input_tokens, avg_output_tokens, 'gpt-4o') * daily_volume * 365
print(f'\nBaseline (all gpt-4o):        ${baseline_cost:,.0f}/yr')
print(f'Optimised (smart routing):    ${optimised_cost:,.0f}/yr')
print(f'Savings:                      ${baseline_cost - optimised_cost:,.0f}/yr ({(1 - optimised_cost/baseline_cost):.0%})')

## ✅ Lab 05 Complete

You have:
- Built **ReAct from scratch** — Reason, Act, Observe, loop
- Traced every LLM call automatically with **MLflow `autolog()`** — zero boilerplate
- Added **custom MLflow spans** around your retrieval and scoring logic
- Evaluated agent output quality with a **GPT-4o-as-judge** (3 dimensions)
- Modelled **cost at scale** and calculated 87% savings with smart routing

**Next:** Lab 06 — Alignment & Fine-tuning